# FFmpeg Worker Loop

This notebook implements a worker loop that:
1. Fetches encoding tasks from `/api/encode/largest`
2. Builds and executes the ffmpeg command
3. Reports results back to `/api/encoded`

In [4]:
import os
import time
import logging
from local_worker_functions import *

## Worker Loop

Main processing loop that continuously fetches tasks, processes them, and reports results

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

__all__ = ["run_local_worker_loop"]


def run_local_worker_loop():
    poll_interval = int(os.environ.get("POLL_INTERVAL", "300"))
    logging.info(f"Worker started. Poll interval: {poll_interval} seconds")
    
    while True:
        try:
            logging.info("=" * 80)
            logging.info("Polling for new task...")
            
            # Step 1: Get largest task from queue
            status, response = get_largest_task()
            
            if status != 200:
                logging.error(f"Failed to get task. Status: {status}, Response: {response}")
                continue
            
            if not response.get('success') or response.get('data') is None:
                logging.info("No tasks available in queue")
                continue
            
            task_data = response['data']
            file_guid = task_data.get('file_guid')
            directory_path = task_data.get('directory_path')
            input_file_name = task_data.get('input_file_name')
            output_file_name = task_data.get('output_file_name')
            before_file_size = task_data.get('before_file_size')
            ffmpeg_command = task_data.get('ffmpeg_string')

            after_file_size = 0

            before_file_size_file_path = os.path.join(directory_path, input_file_name)
            templorary_file_path = os.path.join("/boil/boil_hold/", output_file_name)
            after_file_path = os.path.join(directory_path, output_file_name)

            logging.info(f"Task received: {file_guid}")
            logging.info(f"  Input: {directory_path}/{input_file_name}")
            logging.info(f"  Output: {output_file_name}")
            
            # Step 2: Preflight check - validate file hasn't changed and has integrity
            logging.info("Running preflight check...")
            
            # Step 3: Validate file size hasn't changed
            if not validate_hash(before_file_size_file_path, before_file_size):
                logging.error("Preflight check failed: Video Hash check failed")
                report_encoding_completed(file_guid, 'Failed: Hash mismatch')
                continue
            logging.info("✓ Video integrity check passed")
            
            # Step 4: Validate video integrity
            if not validate_video(before_file_size_file_path):
                logging.error("Preflight check failed: File size mismatch")
                report_encoding_completed(file_guid, 'Failed: Input Integrity')
                continue
            logging.info("✓ Video integrity check passed")
            
            # Step 5: Run ffmpeg encoding
            logging.info("Starting ffmpeg encoding...")
            if not run_ffmpeg(before_file_size_file_path, ffmpeg_command, templorary_file_path):
                logging.error("FFmpeg encoding failed")
                report_encoding_completed(file_guid, 'Failed: FFmpeg failure')
                continue
            logging.info("✓ FFmpeg encoding completed")
            
            # Step 6: Postflight check - validate output video integrity
            logging.info("Running postflight check...")
            if not validate_post_flight_video(before_file_size_file_path):
                logging.error("Postflight check failed: Video integrity check failed")
                report_encoding_completed(file_guid, 'Failed: Postflight Integrity')
                continue
            logging.info("✓ Postflight check passed")

            # Step 7: Get output file size
            logging.info("Getting output file size")
            after_file_size = get_file_size_kb(templorary_file_path)
            if after_file_size == 0:
                logging.error("Postflight file size check failed")
                report_encoding_completed(file_guid, 'Failed: Postflight file size check')
                continue
            logging.info(f"✓ Output file size: {after_file_size} KB")

            # Step 8: Delete source
            logging.info("Processing files...")
            if not delete_file(after_file_path):
                logging.error("Postflight delete source file failed")
                report_encoding_completed(file_guid, 'Failed: Postflight delete source file')
                continue
            logging.info("✓ Postflight check passed")

            # Step 9: Move temporary file to final destination
            logging.info("Moving temporary file to final destination...")
            if not move_file(templorary_file_path, after_file_path):
                logging.error("Failed to move temporary file to final destination")
                report_encoding_completed(file_guid, 'Failed: Move temporary file to final destination')
                continue
            logging.info("✓ File moved successfully")

            # Step 10: Report completion to manager
            logging.info("Reporting completion to manager...")
            report_status, report_response = report_encoding_completed(file_guid, 'encoded', after_file_size)
            if report_status != 200:
                logging.error(f"Failed to report completion. Status: {report_status}, Response: {report_response}")
                # Note: Even if reporting fails, the file has been processed successfully
            else:
                logging.info("✓ Completion reported successfully")
            
            logging.info(f"Task {file_guid} completed successfully!")
            logging.info("=" * 80)
            
        except KeyboardInterrupt:
            logging.info("Worker stopped by user")
            break
        except Exception as e:
            logging.error(f"Unexpected error in worker loop: {type(e).__name__}: {str(e)}")
        
        # Wait before next poll
        time.sleep(poll_interval)


if __name__ == "__main__":
    run_local_worker_loop()


2026-02-01 16:34:33,290 - INFO - Worker started. Poll interval: 300 seconds
2026-02-01 16:34:33,291 - INFO - ================================================================================
2026-02-01 16:34:33,291 - INFO - Polling for new task...
2026-02-01 16:34:33,307 - INFO - No tasks available in queue
2026-02-01 16:34:33,308 - INFO - ================================================================================
2026-02-01 16:34:33,309 - INFO - Polling for new task...
2026-02-01 16:34:33,345 - INFO - No tasks available in queue
2026-02-01 16:34:33,346 - INFO - ================================================================================
2026-02-01 16:34:33,346 - INFO - Polling for new task...
2026-02-01 16:34:33,376 - INFO - No tasks available in queue
2026-02-01 16:34:33,376 - INFO - ================================================================================
2026-02-01 16:34:33,377 - INFO - Polling for new task...
2026-02-01 16:34:33,406 - INFO - No tasks available in qu